# Registered Users: Bronze Ingestion

Sets up the `projectpractice2` catalog and medallion schemas (bronze, silver, gold), then incrementally loads registered users CSV files from the landing zone into `projectpractice2.bronze.registered_users` with COPY INTO, logging each run to `projectpractice2.bronze.ingestion_audit`. Finally, parses user info JSON files into `projectpractice2.bronze.user_info`.

## 1. Catalog setup

In [0]:
%sql 
-- Inspect the medallion2 external location (URL, credential, owner)
-- The URL is used below to derive the bronze/silver/gold storage paths
DESCRIBE EXTERNAL LOCATION medallion2;

In [0]:
%sql
-- Create the project catalog (no-op if it already exists)
CREATE CATALOG IF NOT EXISTS projectpractice2;

In [0]:
%sql
-- Set projectpractice2 as the default catalog for the rest of the session
USE CATALOG projectpractice2;

## 2. Medallion schemas

Derives a storage path for each layer from the `medallion2` external location and creates the bronze, silver and gold schemas with managed locations.

In [0]:
# Derive the bronze layer storage path from the medallion external location
# (external location URL + 'bronze')
bronze_path = (
    spark.sql("""
        DESCRIBE EXTERNAL LOCATION `medallion2`
    """)
    .selectExpr("concat(url, 'bronze') as url")
    .first()["url"]
)

print(bronze_path)

In [0]:
# Create the bronze schema with managed location at the derived storage path
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS bronze
    MANAGED LOCATION '{bronze_path}'
""")

In [0]:
# Derive the silver layer storage path from the medallion external location
# (external location URL + 'silver')
silver_path = (
    spark.sql("""
        DESCRIBE EXTERNAL LOCATION `medallion2`
    """)
    .selectExpr("concat(url, 'silver') as url")
    .first()["url"]
)

print(silver_path)

In [0]:
# Create the silver schema with managed location at the derived storage path
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS silver
    MANAGED LOCATION '{silver_path}'
""")

In [0]:
# Derive the gold layer storage path from the medallion external location
# (external location URL + 'gold')
gold_path = (
    spark.sql("""
        DESCRIBE EXTERNAL LOCATION `medallion2`
    """)
    .selectExpr("concat(url, 'gold') as url")
    .first()["url"]
)

print(gold_path)

In [0]:
# Create the gold schema with managed location at the derived storage path
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS gold
    MANAGED LOCATION '{gold_path}'
""")

## 3. Landing zone check

In [0]:
# List the contents of the raw landing container to confirm source folders exist
display(
    dbutils.fs.ls("abfss://dataingestion@projectpractice2.dfs.core.windows.net/")
)

## 4. Bronze tables

Creates the target `registered_users` table and the `ingestion_audit` log table. The drop cell is only for a full reset.

In [0]:
%sql
-- WARNING: drops the bronze registered_users table so it can be recreated from scratch
-- Only run when a full reset/reload is intended
-- DROP TABLE IF EXISTS projectpractice2.bronze.registered_users;

In [0]:
%sql
-- Create the bronze registered_users table
-- Target of the COPY INTO load from the registeredusers landing folder
-- registration_timestamp is stored as INT (raw epoch value from source)
CREATE TABLE IF NOT EXISTS projectpractice2.bronze.registered_users (
    user_id                  BIGINT,
    device_id                INT,
    mac_address              VARCHAR(100),
    registration_timestamp   INT            -- Unix epoch seconds
);

In [0]:
# Optional: preview a single source CSV file to inspect its schema and values
# Uncomment to run

# file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/registeredusers/1-registered_users_2.csv"
#
# df = (
#     spark.read
#     .format("csv")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load(file_location)
# )
#
# display(df)

In [0]:
%sql
-- Audit table: one row per ingestion run with its outcome and timestamp
CREATE TABLE IF NOT EXISTS projectpractice2.bronze.ingestion_audit (
    run_id          STRING,     -- unique ID per run (UUID)
    source_name     STRING,     -- logical source name, e.g. registered_users
    status          STRING,     -- SUCCESS / FAILED
    load_timestamp  TIMESTAMP   -- when the run finished
)
USING DELTA;

## 5. Ingestion

COPY INTO loads only new files, so re-running is safe. Each run writes one row to the audit table.

In [0]:
import uuid


def load_registered_users1():
    """
    Incrementally load registered users CSV files from the landing zone
    into the bronze table using COPY INTO, then write an audit record.

    COPY INTO is idempotent: files already loaded are skipped on re-runs.
    """

    # ---------- Configuration ----------
    target_table = "projectpractice2.bronze.registered_users"
    audit_table = "projectpractice2.bronze.ingestion_audit"
    source_name = "registered_users"

    source_path = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/registeredusers/"

    # Generate unique run ID
    run_id = str(uuid.uuid4())

    # Row count before the load, used to compute how many rows were added
    rows_before = spark.table(target_table).count()

    try:
        # ---------- Ingestion ----------
        # COPY INTO reads only new files from source_path and casts user_id to BIGINT
        copy_query = f"""
        COPY INTO {target_table}
        FROM (
            SELECT
                CAST(user_id AS BIGINT) AS user_id,
                device_id,
                mac_address,
                registration_timestamp
            FROM '{source_path}'
        )
        FILEFORMAT = CSV
        FORMAT_OPTIONS (
            'header' = 'true',
            'inferSchema' = 'true'
        )
        """

        # Run ingestion
        spark.sql(copy_query)

        # Count after load
        rows_after = spark.table(target_table).count()

        # Number of new rows added by this run
        rows_received = rows_after - rows_before

        status = "SUCCESS"

    except Exception as e:
        # Record the failure in the audit table instead of stopping the notebook
        print("Ingestion failed:", e)
        rows_received = 0
        status = "FAILED"

    # ---------- Audit ----------
    # Write one audit row for this run (success or failure)
    audit_query = f"""
        INSERT INTO {audit_table}
        (
            run_id,
            source_name,
            status,
            load_timestamp
        )
        VALUES
        (
            '{run_id}',
            '{source_name}',
            '{status}',
            current_timestamp()
        )
    """

    spark.sql(audit_query)

    # ---------- Run summary ----------
    print("Run ID:", run_id)
    print("Source:", source_name)
    print("Status:", status)
    print("Rows received:", rows_received)

In [0]:
# Execute the bronze ingestion for registered users
load_registered_users1()

## 6. Validation

In [0]:
%sql 
-- Verify the schema of the bronze registered_users table
DESCRIBE TABLE projectpractice2.bronze.registered_users;

In [0]:
%sql
-- Preview the rows loaded into the bronze table
SELECT *
FROM projectpractice2.bronze.registered_users;

In [0]:
%sql
-- Review the ingestion audit log, most recent runs first
SELECT *
FROM projectpractice2.bronze.ingestion_audit
ORDER BY load_timestamp DESC;

## 7. User info bronze table

Reads the user info JSON files from the `userinfo` landing folder, parses each JSON field from the `value` column, and creates `projectpractice2.bronze.user_info` with flattened address fields, a converted timestamp and a parsed date of birth.

In [0]:
# Preview the raw user info JSON files before building the bronze table
# Single-file path kept for ad hoc inspection of one file:
# file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/2-user_info_1.json"
file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json"

# Read all JSON files in the userinfo folder
df = (
    spark.read
    .format("json")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_location)
)

display(df)

In [0]:
# Source folder containing the user info JSON files
file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json"

# Read all user info JSON files into a DataFrame
df = (
    spark.read
    .format("json")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_location)
)

display(df)

# Register a temp view so the next SQL cell can parse the raw JSON
df.createOrReplaceTempView("df")

In [0]:
from pyspark.sql.functions import col, from_unixtime, to_date

def load_user_info():

    # ============================================================
    # 1. Source Configuration
    # ============================================================

    file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json"
    target_table = "projectpractice2.bronze.user_info"

    # ============================================================
    # 2. Read Source JSON Data
    # ============================================================

    df = (
        spark.read
            .format("json")
            .option("inferSchema", "true")
            .load(file_location)
    )

    (
        df.write
            .format("delta")
            .mode("append")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
    )

    return df

In [0]:
load_user_info()

In [0]:
%sql
-- Preview the rows loaded into the bronze user_info table
SELECT *
FROM projectpractice2.bronze.user_info;

In [0]:
%sql
drop table projectpractice2.bronze.user_info;

In [0]:
%sql
create table if not exists projectpractice2.bronze.user_info;

In [0]:
def load_user_info():

    file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json"
    target_table = "projectpractice2.bronze.user_info"

    # Read JSON files
    df = (
        spark.read
            .format("json")
            .option("inferSchema", "true")
            .load(file_location)
    )

    # Write to Delta table
    (
        df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
    )

    return df


df = load_user_info()

display(df)

In [0]:
%sql
create table if not exists projectpractice2.bronze.user_info;

In [0]:
%sql
COPY INTO projectpractice2.bronze.user_info
        FROM (
            SELECT
                *
            FROM 'abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json'
        )
        FILEFORMAT = JSON
        COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
select * from projectpractice2.bronze.user_info;

In [0]:
%sql
create table if not exists projectpractice2.bronze.user_info_temp;

In [0]:
%sql
COPY INTO projectpractice2.bronze.user_info_temp
        FROM (
            SELECT
                *
            FROM 'abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json'
        )
        FILEFORMAT = JSON
        COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
select * from projectpractice2.bronze.user_info_temp

In [0]:
file_location4 = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/datelookup/6-date-lookup.json"

df4 = (
     spark.read
     .format("json")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location4)
)

display(df4)

In [0]:
df4.createOrReplaceTempView("df4")

In [0]:
%sql
create table projectpractice2.bronze.date_lookup_table as
select * from df4;

In [0]:
%sql
create table if not exists projectpractice2.silver.date_lookup_table as
select cast(date as date) as date, dayofmonth, dayofweek, dayofyear, month, week, week_part, year from projectpractice2.bronze.date_lookup_table;

In [0]:
%sql
select * from projectpractice2.silver.date_lookup_table;